In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/NLP_project/processed")
test = pd.read_csv(BASE_DIR / "test_seg.csv")
print(test.shape)

Mounted at /content/drive
(3166, 4)


In [ ]:
subset = (
    test.groupby(["sentiment", "topic"], group_keys=False)
    .apply(lambda x: x.sample(frac=400/len(test), random_state=42))
)
subset = subset.reset_index(drop=True)
print(f"Subset size: {len(subset)}")
print(subset["sentiment"].value_counts())
print(subset["topic"].value_counts())

subset.to_csv(BASE_DIR / "gemini_subset.csv", index=False, encoding="utf-8-sig")

Subset size: 400
sentiment
positive    201
negative    178
neutral      21
Name: count, dtype: int64
topic
lecturer            289
training_program     73
others               20
facility             18
Name: count, dtype: int64


/tmp/ipykernel_3182/3242971313.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(frac=400/len(test), random_state=42))


In [ ]:
import pandas as pd


test = pd.read_csv(BASE_DIR / "test_seg.csv")

# Lấy mẫu phân tầng theo sentiment x topic để giữ tỷ lệ đại diện
subset = (
    test.groupby(["sentiment", "topic"], group_keys=False)
    .apply(lambda x: x.sample(frac=400/len(test), random_state=42))
)
subset = subset.reset_index(drop=True)
print(f"Subset size: {len(subset)}")
print(subset["sentiment"].value_counts())
print(subset["topic"].value_counts())

subset.to_csv(BASE_DIR / "gemini_subset.csv", index=False, encoding="utf-8-sig")

Subset size: 400
sentiment
positive    201
negative    178
neutral      21
Name: count, dtype: int64
topic
lecturer            289
training_program     73
others               20
facility             18
Name: count, dtype: int64


/tmp/ipykernel_3182/3467021568.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(frac=400/len(test), random_state=42))


In [ ]:
%pip install -U google-genai --quiet

from google import genai

# Dùng Colab Secrets thay vì hardcode key (an toàn hơn)
from google.colab import userdata
client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents="Xin chào"
)
print(response.text)

Xin chào! Tôi có thể giúp gì cho bạn hôm nay?


In [ ]:
PROMPT_TEMPLATE = """Bạn là một hệ thống phân loại phản hồi sinh viên đại học tiếng Việt.
Với câu phản hồi dưới đây, hãy xác định:
1. sentiment: chỉ chọn 1 trong 3 giá trị "negative", "neutral", "positive"
2. topic: chỉ chọn 1 trong 4 giá trị "lecturer", "training_program", "facility", "others"

Định nghĩa:
- lecturer: nhận xét về giảng viên (cách dạy, thái độ, chuyên môn)
- training_program: nhận xét về nội dung môn học, chương trình đào tạo, bài tập
- facility: nhận xét về cơ sở vật chất (phòng học, thiết bị, phòng lab)
- others: không thuộc 3 nhóm trên

Chỉ trả lời bằng JSON, không thêm giải thích:
{{"sentiment": "...", "topic": "..."}}

Câu: "{sentence}"
"""

In [ ]:
import re

def parse_retry_delay(error_msg, default=15):
    """Đọc chính xác thời gian cần chờ từ thông báo lỗi 429 của Google"""
    match = re.search(r"retry in (\d+\.?\d*)s", error_msg, re.IGNORECASE)
    if match:
        return float(match.group(1)) + 2  # +2s buffer an toàn
    return default

def classify_with_gemini(sentence, max_retries=4):
    prompt = PROMPT_TEMPLATE.format(sentence=sentence)
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model="gemini-3.1-flash-lite",
                contents=prompt,
            )
            result = extract_json(response.text)
            sentiment = result.get("sentiment", "").strip().lower()
            topic = result.get("topic", "").strip().lower()
            if sentiment in ["negative", "neutral", "positive"] and \
               topic in ["lecturer", "training_program", "facility", "others"]:
                return sentiment, topic
            else:
                print(f"  Nhãn không hợp lệ: {result}, thử lại...")
        except Exception as e:
            wait = parse_retry_delay(str(e))
            print(f"  Lỗi (attempt {attempt+1}): {type(e).__name__} -> chờ {wait:.0f}s")
            time.sleep(wait)
    return None, None

In [ ]:
import os

# Kiểm tra xem đã có file tạm chưa (để resume nếu bị ngắt giữa chừng)
temp_path = BASE_DIR / "gemini_results_temp.csv"
if temp_path.exists():
    done_df = pd.read_csv(temp_path)
    n_done = len(done_df)
    gemini_sentiments = done_df["gemini_sentiment"].tolist()
    gemini_topics = done_df["gemini_topic"].tolist()
    print(f"Tiếp tục từ câu thứ {n_done}")
else:
    n_done = 0
    gemini_sentiments = []
    gemini_topics = []

subset = pd.read_csv(BASE_DIR / "gemini_subset.csv")

for pos, (_, row) in enumerate(tqdm(subset.iloc[n_done:].iterrows(), total=len(subset) - n_done)):
    sent, topic = classify_with_gemini(row["sentence"])
    gemini_sentiments.append(sent)
    gemini_topics.append(topic)
    time.sleep(4)

    current_count = n_done + pos + 1
    if current_count % 20 == 0 or current_count == len(subset):
        temp_df = subset.iloc[:current_count].copy()
        temp_df["gemini_sentiment"] = gemini_sentiments
        temp_df["gemini_topic"] = gemini_topics
        temp_df.to_csv(temp_path, index=False, encoding="utf-8-sig")

subset["gemini_sentiment"] = gemini_sentiments
subset["gemini_topic"] = gemini_topics
subset.to_csv(BASE_DIR / "gemini_results.csv", index=False, encoding="utf-8-sig")
print(f"Hoàn tất. Số câu thất bại: {subset['gemini_sentiment'].isna().sum()}")

  0%|          | 0/400 [00:00<?, ?it/s]

  Lỗi (attempt 1): ServerError -> chờ 15s


 40%|████      | 162/400 [15:01<27:11,  6.86s/it]

  Lỗi (attempt 1): ServerError -> chờ 15s


 88%|████████▊ | 354/400 [32:27<04:17,  5.59s/it]

  Lỗi (attempt 1): ServerError -> chờ 15s


100%|██████████| 400/400 [36:49<00:00,  5.52s/it]

Hoàn tất. Số câu thất bại: 0


In [ ]:
from sklearn.metrics import classification_report, accuracy_score, f1_score

results = pd.read_csv(BASE_DIR / "gemini_results.csv")

print("=== Gemini - Sentiment ===")
print(classification_report(results["sentiment"], results["gemini_sentiment"]))

print("\n=== Gemini - Topic ===")
print(classification_report(results["topic"], results["gemini_topic"]))

=== Gemini - Sentiment ===
              precision    recall  f1-score   support

    negative       0.97      0.85      0.91       178
     neutral       0.34      0.67      0.45        21
    positive       0.96      0.96      0.96       201

    accuracy                           0.90       400
   macro avg       0.76      0.83      0.77       400
weighted avg       0.93      0.90      0.91       400


=== Gemini - Topic ===
                  precision    recall  f1-score   support

        facility       0.89      0.94      0.92        18
        lecturer       0.96      0.89      0.92       289
          others       0.44      0.20      0.28        20
training_program       0.58      0.82      0.68        73

        accuracy                           0.84       400
       macro avg       0.72      0.71      0.70       400
    weighted avg       0.86      0.84      0.85       400



In [ ]:
ROOT_DIR = Path("/content/drive/MyDrive/NLP_project")

phobert_full = pd.read_csv(ROOT_DIR / "test_predictions.csv")
phobert_subset = phobert_full[phobert_full["sentence"].isin(results["sentence"])].copy()

print(f"Số câu khớp trong PhoBERT: {len(phobert_subset)}/{len(results)}")

print("\n=== PhoBERT (trên cùng subset 400 câu) - Sentiment ===")
print(classification_report(phobert_subset["true_sentiment"], phobert_subset["pred_sentiment"]))

print("\n=== PhoBERT (trên cùng subset 400 câu) - Topic ===")
print(classification_report(phobert_subset["true_topic"], phobert_subset["pred_topic"]))

Số câu khớp trong PhoBERT: 400/400

=== PhoBERT (trên cùng subset 400 câu) - Sentiment ===
              precision    recall  f1-score   support

    negative       0.93      0.97      0.95       178
     neutral       0.75      0.43      0.55        21
    positive       0.94      0.95      0.94       201

    accuracy                           0.93       400
   macro avg       0.87      0.78      0.81       400
weighted avg       0.93      0.93      0.93       400


=== PhoBERT (trên cùng subset 400 câu) - Topic ===
                  precision    recall  f1-score   support

        facility       0.94      0.94      0.94        18
        lecturer       0.94      0.95      0.94       289
          others       0.88      0.35      0.50        20
training_program       0.75      0.82      0.78        73

        accuracy                           0.90       400
       macro avg       0.88      0.77      0.79       400
    weighted avg       0.90      0.90      0.89       400



In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report

# Load lại dữ liệu train + subset Gemini
train = pd.read_csv(BASE_DIR / "train_seg.csv")
gemini_results = pd.read_csv(BASE_DIR / "gemini_results.csv")

# Cần sentence_seg cho subset để predict bằng TF-IDF (đã tách từ sẵn từ lúc tạo subset)
# Nếu gemini_subset.csv gốc có sentence_seg, dùng lại; nếu không, tách từ lại nhanh
if "sentence_seg" not in gemini_results.columns:
    %pip install pyvi --quiet
    from pyvi import ViTokenizer
    gemini_results["sentence_seg"] = gemini_results["sentence"].apply(ViTokenizer.tokenize)

# Train lại SVM (nhanh, vài giây)
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=2)
X_train = vectorizer.fit_transform(train["sentence_seg"])
X_subset = vectorizer.transform(gemini_results["sentence_seg"])

svm_sentiment = LinearSVC(class_weight="balanced", random_state=42)
svm_sentiment.fit(X_train, train["sentiment"])
svm_sent_preds = svm_sentiment.predict(X_subset)

svm_topic = LinearSVC(class_weight="balanced", random_state=42)
svm_topic.fit(X_train, train["topic"])
svm_topic_preds = svm_topic.predict(X_subset)

gemini_results["svm_sentiment"] = svm_sent_preds
gemini_results["svm_topic"] = svm_topic_preds
gemini_results.to_csv(BASE_DIR / "three_way_comparison.csv", index=False, encoding="utf-8-sig")

print("=== SVM (trên cùng subset 400 câu) - Sentiment ===")
print(classification_report(gemini_results["sentiment"], svm_sent_preds))

print("\n=== SVM (trên cùng subset 400 câu) - Topic ===")
print(classification_report(gemini_results["topic"], svm_topic_preds))

=== SVM (trên cùng subset 400 câu) - Sentiment ===
              precision    recall  f1-score   support

    negative       0.91      0.97      0.93       178
     neutral       0.36      0.24      0.29        21
    positive       0.93      0.91      0.92       201

    accuracy                           0.90       400
   macro avg       0.73      0.70      0.71       400
weighted avg       0.89      0.90      0.89       400


=== SVM (trên cùng subset 400 câu) - Topic ===
                  precision    recall  f1-score   support

        facility       0.84      0.89      0.86        18
        lecturer       0.94      0.91      0.92       289
          others       0.44      0.35      0.39        20
training_program       0.61      0.70      0.65        73

        accuracy                           0.84       400
       macro avg       0.71      0.71      0.71       400
    weighted avg       0.85      0.84      0.85       400

